In [1]:
import cv2
import numpy as np
from pdf2image import convert_from_path
from scipy.ndimage import rotate
from PIL import Image
import pytesseract

# Function to correct skew
def correct_skew(image, delta=1, limit=12):
    def determine_score(arr, angle):
        data = rotate(arr, angle, reshape=False, order=0)
        histogram = np.sum(data, axis=1, dtype=float)
        score = np.sum((histogram[1:] - histogram[:-1]) ** 2, dtype=float)
        return histogram, score

    gray = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 41, 15)
    scores = []
    angles = np.arange(-limit, limit + delta, delta)
    
    for angle in angles:
        histogram, score = determine_score(thresh, angle)
        scores.append(score)
    
    best_angle = angles[scores.index(max(scores))]
    return best_angle

# Function to deskew image
def deskew_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated

# Function to detect text orientation using Tesseract
def detect_orientation(image):
    image = np.array(image)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    osd = pytesseract.image_to_osd(rgb_image)
    rotate_angle = 0
    if "Rotate: 180" in osd:
        rotate_angle = 180
    return rotate_angle

# Path to the PDF file
pdf_path = '/home/emanmunir/detection/test2/pdf to image/scan-rotated.pdf'
# Convert PDF to images
images = convert_from_path(pdf_path)

# Process images and save to a new PDF
pdf_images = []
for i, image in enumerate(images):
    rotate_angle = detect_orientation(image)
    if rotate_angle == 180:
        image = image.rotate(180, expand=True)
    angle = correct_skew(image)
    deskewed_image = deskew_image(np.array(image), angle)
    final_image = Image.fromarray(deskewed_image)
    pdf_images.append(final_image.convert('RGB'))

# Save all images into a single PDF
pdf_images[0].save('/home/emanmunir/detection/test2/pdf to image/corrected.pdf', save_all=True, append_images=pdf_images[1:])

print("PDF pages have been converted, deskewed, and saved as a single PDF.")


PDF pages have been converted, deskewed, and saved as a single PDF.


In [9]:
import fitz  # PyMuPDF

# Function to get the dimensions of the first page of a PDF
def get_pdf_dimensions(pdf_path):
    pdf = fitz.open(pdf_path)
    first_page = pdf[0]
    dimensions = first_page.rect
    pdf.close()
    return dimensions

# Example usage
source_pdf_path = '/home/emanmunir/detection/test2/pdf to image/small-pdf-boxes.pdf'
target_pdf_path = '/home/emanmunir/detection/test2/pdf to image/corrected.pdf'

source_dimensions = get_pdf_dimensions(source_pdf_path)
target_dimensions = get_pdf_dimensions(target_pdf_path)

print(f"Source PDF dimensions: {source_dimensions}")
print(f"Target PDF dimensions: {target_dimensions}")


Source PDF dimensions: Rect(0.0, 0.0, 612.0, 792.0)
Target PDF dimensions: Rect(0.0, 0.0, 1653.0, 2339.0)


In [7]:
import fitz  # PyMuPDF
from PIL import Image
import io

def extract_images(pdf_path):
    doc = fitz.open(pdf_path)
    images = []
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        image_list = []
        for img in page.get_images(full=True):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))
            image_list.append((xref, image))
        images.append(image_list)
    return images

def replace_image(doc, page_num, xref, new_image):
    img_bytes = io.BytesIO()
    new_image.save(img_bytes, format="PNG")
    img_bytes.seek(0)
    img_xref = doc.add_image(img_bytes)
    doc[page_num].replace_image(xref, img_xref)

def sync_pdf_images(pdf1_path, pdf2_path, output_path):
    doc1 = fitz.open(pdf1_path)
    doc2 = fitz.open(pdf2_path)

    images1 = extract_images(pdf1_path)
    images2 = extract_images(pdf2_path)

    for page_num in range(min(len(images1), len(images2))):
        for (xref1, img1), (xref2, img2) in zip(images1[page_num], images2[page_num]):
            replace_image(doc2, page_num, xref2, img1)
            print(f"Replaced image on page {page_num + 1}")

    # Save the synchronized PDF
    doc2.save(output_path)

pdf1_path = "small-pdf-boxes.pdf"
pdf2_path = "corrected.pdf"
output_path = "synced_output.pdf"

sync_pdf_images(pdf1_path, pdf2_path, output_path)
